# Arrays, tidy tables, and validated joins

**P0 Essential · D2 Independent · 90 minutes**

In [ ]:
import pandas as pd

from pathlib import Path

def locate(relative: str, local_name: str | None = None) -> Path:
    candidates = []
    if local_name:
        candidates.append(Path.cwd() / local_name)
    candidates.extend(root / relative for root in [Path.cwd(), *Path.cwd().parents])
    for candidate in candidates:
        if candidate.exists():
            return candidate
    raise FileNotFoundError(
        f"Cannot find {relative}. Run from the course clone or place the downloaded data beside this notebook."
    )


In [ ]:
obs_path = locate('datasets/teaching/air-quality/observations.csv', 'observations.csv')
stations_path = locate('datasets/teaching/air-quality/stations.csv', 'stations.csv')
observations = pd.read_csv(obs_path)
stations = pd.read_csv(stations_path)
observations.shape, stations.shape

## Predict before running

What should uniquely identify one hourly station observation? What join cardinality connects observations to the station registry?

## Task

Construct a timestamp, validate the observation key, produce a tidy pollutant table, and join the registry without changing the observation-key set.

In [ ]:
def prepare_tables(observations: pd.DataFrame, stations: pd.DataFrame):
    frame = observations.copy()
    frame['timestamp'] = pd.to_datetime(frame[['year', 'month', 'day', 'hour']])
    key = ['station', 'timestamp']
    if frame.duplicated(key).any():
        raise ValueError('Observation key is not unique')
    pollutants = frame.melt(
        id_vars=key, value_vars=['PM2.5', 'PM10', 'NO2'],
        var_name='pollutant', value_name='concentration'
    )
    before = set(map(tuple, frame[key].itertuples(index=False, name=None)))
    joined = frame.merge(stations, on='station', how='left', validate='many_to_one', indicator=True)
    after = set(map(tuple, joined[key].itertuples(index=False, name=None)))
    if before != after or not joined['_merge'].eq('both').all():
        raise ValueError('Join changed or failed to match observation keys')
    return joined.drop(columns='_merge'), pollutants

In [ ]:
joined, pollutants = prepare_tables(observations, stations)
assert joined[['station', 'timestamp']].duplicated().sum() == 0
assert len(joined) == len(observations)
assert set(pollutants['pollutant']) == {'PM2.5', 'PM10', 'NO2'}
assert len(pollutants) == 3 * len(observations)
joined[['station', 'timestamp', 'PM2.5', 'source_file']].head()

## Transfer

Create a daily station summary. State its unit and key, preserve missingness, and assert uniqueness. Explain why row-count equality alone would not prove join correctness.

**Instructor note.** Require students to compare key sets, not only row counts, and to explain storage dtype versus semantic role.